In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
import torch.autograd as autograd
from sklearn.metrics import accuracy_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedKFold

device = torch.device("cuda:3")

class TICFM(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, vocab_size, num_layers, num_heads, max_len, dropout, pad_idx, batch_size, peptide_feat_dim) -> None:
        super(TICFM, self).__init__()
        
        self.emb = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        
        self.encoder = nn.GRU(embedding_dim, hidden_dim, num_layers=num_layers, 
                              bidirectional=False, batch_first=True, dropout=dropout)

        self.hidden_to_latent = nn.Linear(hidden_dim, 512)
        

        self.decoder = nn.GRU(512, 512, num_layers=num_layers, 
                              bidirectional=False, batch_first=True, dropout=dropout)
        

        self.peptide_feat_dim = peptide_feat_dim
        

        self.fc = nn.Sequential( 
            nn.LayerNorm(512 + peptide_feat_dim),  
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512 + peptide_feat_dim, 2)  
        )

    def forward(self, x, peptide_feats):
        embedded = self.emb(x)

        encoder_out, _ = self.encoder(embedded)
        
        latent = self.hidden_to_latent(encoder_out[:, -1, :])
        
        decoder_out, _ = self.decoder(latent.unsqueeze(1).repeat(1, x.size(1), 1)) 
        
        peptide_feats = peptide_feats.to(device)
        combined_features = torch.cat((decoder_out[:, -1, :], peptide_feats), dim=1)
        
        out = self.fc(combined_features)  
        return out